In [1]:
# For Colab
import os
from google.colab import userdata, drive

drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

Mounted at /content/drive


In [2]:
!git clone https://github.com/jjjh02/AmoRe_crm_generator.git
!git checkout jinhyeok
%cd AmoRe_crm_generator/finetuning
!pwd

Cloning into 'AmoRe_crm_generator'...
remote: Enumerating objects: 649, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 649 (delta 97), reused 101 (delta 76), pack-reused 508 (from 1)
Receiving objects: 100% (649/649), 6.49 MiB | 16.24 MiB/s, done.
Resolving deltas: 100% (383/383), done.
fatal: not a git repository (or any of the parent directories): .git
/content/AmoRe_crm_generator/finetuning
/content/AmoRe_crm_generator/finetuning


In [9]:
import os
import json
import time
import re
import difflib
from textwrap import dedent
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

# DPO_DATASET_DIR = "./finetuning_data/crm-dpo-dataset/"
# SFT_DATASET_DIR = "./finetuning_data/crm-sft-dataset/"

# For Colab
PROJECT_DIR = "/content/drive/MyDrive/LikeLion/Small Challenge"
DPO_DATASET_DIR = os.path.join(PROJECT_DIR, "dataset_dpo")
SFT_DATASET_DIR = os.path.join(PROJECT_DIR, "dataset_sft")

PERSONAS_PATH = "./data/personas.json"
os.makedirs(SFT_DATASET_DIR, exist_ok=True)

persona_tokens = []
if os.path.exists(PERSONAS_PATH):
    with open(PERSONAS_PATH, "r", encoding="utf-8") as f:
        personas = json.load(f)
    persona_tokens = sorted({p.get("name", "") for p in personas if p.get("name")})


# 기존 DPO 데이터셋 파일 읽기
with open(os.path.join(DPO_DATASET_DIR, "cycle_01_v5.json"), "r") as f:
    data = json.load(f)
    print(f"로드된 DPO 데이터 수 : {len(data)}")
    print(f"DPO 데이터 컬럼 들 : {list(data[0].keys())}")
    print(data[:10])
    print(type(data), type(f))

# SFT용 추가 프롬프트 정의 (tone_correction.py 기반 업데이트)
output_template = dedent("""
아래 출력 규칙과 컨텍스트를 바탕으로 CRM 메시지를 재작성하라.

## 핵심 재작성 원칙

### 1) 형식 및 길이 준수
- 제목은 한 줄 분량(20~25자)으로 간결하게 작성
- 본문은 두 줄~세 줄 분량(80~120자)으로 작성
- 본문의 마지막 한 줄은 반드시 CTA(Call-to-Action) 문장이어야 한다
- 출력 형식은 반드시 '[제목]'과 '[본문]' 레이블을 사용

### 2) Persona / Stage / Tone Guide 반영
- 컨텍스트에 주어진 Persona, Stage, Vibe, Tone Guide를 반드시 반영하라
- Tone Guide의 어조(존칭/반말, 정중/캐주얼 등)를 정확히 따라야 한다
- Stage(할인/신제품런칭/재구매 등)에 맞는 메시지 방향성을 유지하라
- Vibe(2030/4060)에 맞는 표현 수준을 사용하라

### 3) Event 정합성 검증
- 컨텍스트의 Event 항목을 반드시 확인하라
- Event가 "없음"이면: 본문에 이벤트/혜택/쿠폰/할인/적립 등을 언급하지 마라 (할루시네이션 금지)
- Event가 있으면: 해당 이벤트를 본문에 반드시 자연스럽게 포함하라 (누락 금지)
- 원문에 컨텍스트에 없는 이벤트가 언급되어 있다면 제거하라

## 금지 요소
- 영어 사용 금지: 브랜드/제품 고유명 제외, 한국어로만 작성
  (금지 예시: everyday, daily, routine, essential, ultimate, available 등)
- 페르소나/개인정보 직접 호명 금지
  (금지 예시: CL_01님, 오프라인파 스마트 쇼퍼님, user_name 등)
- 메타 표현/특수문자 금지: { } ( ) * : ; # > < % _ 등
- JSON/코드 조각 금지: ```json, ### 등
- 과도한 이모지/구분선 금지

## 표현 수위 가이드
- 허용: '도움을 줄 수 있어요', '편안하게 느껴질 수 있어요', '부담 없이 사용하기 좋아요'
- 지양: '완벽 개선', '즉시 효과', '100% 보장', '치료' 등 의학적/단정 표현

[출력 형식]
[제목] 제목 내용
[본문] 본문 내용
""")

MODEL_NAME = "gpt-4o-mini"
BATCH_SIZE = 1
MAX_OUTPUT_TOKENS = 1024
RETRY_LIMIT = 3

DEBUG = True
DEBUG_MAX_CHARS = 500

# 글자 수 기준 업데이트 (tone_correction.py 기반)
MIN_TITLE_LEN = 5  # 최소 제목 길이 (20-25자 권장)
MAX_TITLE_LEN = 40  # 최대 제목 길이
MIN_BODY_LEN = 40   # 최소 본문 길이 (80-120자 권장)
MAX_BODY_LEN = 150  # 최대 본문 길이

client = OpenAI()

def _debug(message):
    if DEBUG:
        print(message)

def _response_text(response):
    """Chat Completions API 응답에서 텍스트 추출"""
    try:
        return response.choices[0].message.content or ""
    except Exception:
        return ""

def rule_based_clean(text, persona_tokens):
    """규칙 기반 텍스트 정제 (tone_correction.py 금지 요소 기반)

    [제목]/[본문] 레이블과 쉼표는 유지함
    """
    if not text:
        return text
    cleaned = text
    # JSON/코드 조각 제거
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "")
    cleaned = re.sub(r"(?m)^\s*(json|text)\s*$", "", cleaned)
    # 메타 표현/특수문자 제거 (대괄호[], 쉼표는 유지)
    cleaned = cleaned.translate(str.maketrans("", "", "{}"))
    cleaned = re.sub(r"\"?title\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\"?body\"?\s*:", "", cleaned, flags=re.IGNORECASE)
    # cleaned = re.sub(r"CTA\s*:", "", cleaned)
    # 페르소나 직접 호명 제거
    if persona_tokens:
        pattern = r"(?<!\w)(?:" + "|".join(map(re.escape, persona_tokens)) + r")(?:님)?(?!\w)"
        cleaned = re.sub(pattern, "", cleaned)
    cleaned = re.sub(r"\S+님을\s*위해", "", cleaned)
    cleaned = re.sub(r"\S+분들께", "", cleaned)
    # 금지된 영어 표현 제거 (브랜드/제품명 제외)
    forbidden_english = [
        "everyday", "daily", "routine", "essential", "ultimate",
        "available", "special", "premium", "luxury", "exclusive"
    ]
    for word in forbidden_english:
        cleaned = re.sub(rf"\b{word}\b", "", cleaned, flags=re.IGNORECASE)
    # 공백 정리
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()

def extract_removed_parts(original, cleaned, limit=5):
    matcher = difflib.SequenceMatcher(None, original, cleaned)
    parts = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ("delete", "replace"):
            removed = original[i1:i2]
            removed = re.sub(r"\s+", " ", removed).strip()
            if removed:
                if len(removed) > 80:
                    removed = removed[:77] + "..."
                parts.append(removed)
    seen = set()
    uniq = []
    for part in parts:
        if part not in seen:
            seen.add(part)
            uniq.append(part)
    return uniq[:limit]

def _len_in_range(text, min_len, max_len):
    """글자 수가 범위 내에 있는지 확인"""
    if text is None:
        return False
    compact = re.sub(r"\s+", "", text)
    return min_len <= len(compact) <= max_len

def _min_len_ok(text, min_len):
    if text is None:
        return False
    compact = re.sub(r"\s+", "", text)
    return len(compact) >= min_len

def normalize_refined(text, original_draft=""):
    """GPT 출력을 정규화하여 [제목]/[본문] 형식으로 변환

    길이 검증 실패 시에도 데이터를 제외하지 않고 원본 또는 최선의 결과를 반환
    """
    if not text:
        # GPT 출력이 없으면 원본 draft 반환
        return original_draft if original_draft else ""

    cleaned = text.strip()
    cleaned = re.sub(r"```[a-zA-Z]*", "", cleaned)
    cleaned = cleaned.replace("```", "").strip()
    title = None
    body = None

    # Try JSON extraction first
    if "{" in cleaned and "}" in cleaned:
        start = cleaned.find("{")
        end = cleaned.rfind("}") + 1
        try:
            obj = json.loads(cleaned[start:end])
            if isinstance(obj, dict):
                title = str(obj.get("title", "")).strip() or None
                body = str(obj.get("body", "")).strip() or None
        except Exception:
            pass

    # Fallback to label lines
    if title is None or body is None:
        lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
        for line in lines:
            # [제목] 또는 제목: 형식 파싱
            if line.startswith("[제목]"):
                title = line.split("[제목]", 1)[1].strip() or title
            elif line.startswith("제목:"):
                title = line.split("제목:", 1)[1].strip() or title
            elif line.lower().startswith("title:"):
                title = line.split(":", 1)[1].strip() or title
            # [본문] 또는 본문: 형식 파싱
            elif line.startswith("[본문]"):
                body = line.split("[본문]", 1)[1].strip() or body
            elif line.startswith("본문:"):
                body = line.split("본문:", 1)[1].strip() or body
            elif line.lower().startswith("body:"):
                body = line.split(":", 1)[1].strip() or body
        if title is None and lines:
            title = lines[0]
        if body is None and len(lines) > 1:
            body = " ".join(lines[1:])

    # 파싱 실패 시 원본 반환
    if not title and not body and original_draft:
        return original_draft

    # 최소한의 결과라도 반환
    if not title:
        title = "제목"
    if not body:
        body = "본문"

    result = []
    result.append(f"[제목] {title}")
    result.append(f"[본문] {body}")
    return "\n".join(result).strip()

def refine_message(context, draft, output_rules):
    system = output_rules.strip()
    user = f"{context}\n\n{draft}\n"
    last_error = None
    for attempt in range(RETRY_LIMIT):
        try:
            _debug(f"[GPT] attempt={attempt + 1}/{RETRY_LIMIT} model={MODEL_NAME}")
            _debug(f"[GPT] system_len={len(system)} user_len={len(user)}")
            _debug(f"[GPT] system_preview=\n{system[:DEBUG_MAX_CHARS]}")
            _debug(f"[GPT] user_preview=\n{user[:DEBUG_MAX_CHARS]}")
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                max_tokens=MAX_OUTPUT_TOKENS,
            )
            text = _response_text(response).strip()
            text = normalize_refined(text, draft)  # 원본 draft 전달
            _debug(f"[GPT] output_text_len={len(text)}")
            _debug(f"[GPT] output_text_preview=\n{text[:DEBUG_MAX_CHARS]}")
            if text:
                return text
        except Exception as exc:
            last_error = exc
            _debug(f"[GPT] error={type(exc).__name__}: {exc}")
            time.sleep(2 ** attempt)
    # 모든 재시도 실패 시 원본 반환
    if draft:
        return draft
    if last_error:
        raise last_error
    return ""


로드된 DPO 데이터 수 : 2160
DPO 데이터 컬럼 들 : ['prompt', 'chosen', 'rejected', 'best_index', 'rejected_index', 'reason_best', 'reason_rejected']
[{'prompt': '[컨텍스트]\n- Persona: 백화점 기반 실속파 다구매자\n- Stage: 할인\n- Brand/Product: 이니스프리 / 그린티 세라마이드 밀크 에센스\n- Event: 첫 구매 20% 할인 쿠폰\n- Vibe: 4060\n- Tone Guide: 존칭 중심의 정중한 문장으로 안정감과 신뢰를 전달하며, 차분한 표현으로 품질과 편의성을 강조합니다.\n- Context Example: 가족용 보디케어나 비타민 등 여러 품목을 한꺼번에 대량 쇼핑할 때', 'chosen': '[제목]  \n수분 에너지,속급력의 새로운 시작  \n\n[본문]  \n이니스프리 그린티 세라마이드 밀크 에센스는 피부에 흐르는 강력한 수분을 직접 흡수해줍니다. 히알루론산보다 빠른속급력으로 건조함 없이 촉촉함을 유지하죠. 백화점 stores에서 검증된 실적을 바탕으로 한 합리적인 선택입니다.  \n\n첫 구매 고객 전용 990원 체험 증정 이벤트 진행 중입니다. 한정 수량이므로 서둘러 참여하실까요?  \n\n[CTA]  \n지금 바로 체험해보고 효과 확인하세요.', 'rejected': '[제목]  \n그린티 세라마이드 밀크 에센스, 피부 마시는 powerful hydration!  \n\n[본문]  \n이니스프리 그린티 세라마이드 밀크 에센스는속습 종결템으로 히알루론산보다 강한 보습력을 자랑합니다. 백화점 실속파 고객님들께서는 가족용 제품 대량 구매 시 이 제품으로 효율성을 극대화하시죠. 다양한 라인을 한 번에 활용해 비용을 절감하면서도 피부 건강을 위한 필수 아이템으로 추천합니다. 지금 특별 할인 기간 동안 추가 혜택까지 함께 받아보실 수 있습니다.', 'best_index': 2, 'rejected_index': 0,

In [ ]:
# 데이터 갯수 확인 이후 남은 데이터 refine
try:
    with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v6.jsonl"), "r") as f:
        remain_data_num = sum(1 for _ in f)
        print(f"저장된 SFT 데이터 수 : {remain_data_num}")
except:
    remain_data_num = 0

# SFT 프롬프트용 출력 규칙 (tone_correction.py 기반 업데이트)
output_template_for_sft = dedent("""
## 필수 출력 형식 규칙
1) 제목은 한 줄 분량(20~25자)으로 간결하게 작성
2) 본문은 두 줄~세 줄 분량(80~120자)으로 작성
3) 본문 마지막은 CTA(Call-to-Action) 문장 필수
4) 출력 형식: [제목], [본문] 레이블 사용

## 금지 요소
- 영어 사용 금지 (브랜드/제품 고유명 제외)
- 페르소나/개인정보 직접 호명 금지
- 메타 표현/특수문자/이모지 금지

[출력 형식]
[제목] 제목 내용
[본문] 본문 내용
""")

total_batches = (len(data[remain_data_num:]) + BATCH_SIZE - 1) // BATCH_SIZE
for batch_index in range(total_batches):
    refined_data = []
    start = batch_index * BATCH_SIZE + remain_data_num
    batch = data[start : start + BATCH_SIZE]
    print(f"Batch {batch_index + 1}/{total_batches}")

    for idx_in_batch, example in enumerate(batch):
        sample_index = start + idx_in_batch + 1
        context = example.get("prompt", "")
        draft = example.get("chosen", "")
        cleaned = rule_based_clean(draft, persona_tokens)
        if cleaned != draft:
            removed_parts = extract_removed_parts(draft, cleaned)
            if removed_parts:
                print(f"[정제됨] {sample_index}번: " + ", ".join(removed_parts))
            else:
                print(f"[정제됨] {sample_index}번: 변경 감지(공백/형식)")

        # refine_message 호출 (실패 시에도 원본 또는 최선 결과 반환)
        refined = refine_message(context, cleaned, output_template)

        # 길이 부족 시에도 데이터 제외하지 않고 포함
        # refined가 비어있으면 cleaned를 사용
        if not refined:
            print(f"[경고] {sample_index}번: GPT 출력 실패, 정제된 원본 사용")
            refined = cleaned if cleaned else draft

        refined_data.append({**example, "chosen_refined": refined})

    with open(os.path.join(SFT_DATASET_DIR, "cycle_01_v6.jsonl"), "a") as f:
        for example in refined_data:
            prompt = example.get("prompt", "")
            chosen = example.get("chosen_refined", example.get("chosen", ""))
            prompt = "다음 조건에 맞는 CRM 메시지를 작성하세요. " + prompt + f" {output_template_for_sft.replace(chr(10), ' ')}"
            f.write(json.dumps({"prompt": prompt, "chosen": chosen}, ensure_ascii=False) + "\n")

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
- 출력 형식은 반드시 '[제목]'과 '[본문]' 레이블을 사용

### 2) Persona / Stage / Tone Guide 반영
- 컨텍스트에 주어진 Persona, Stage, Vibe, Tone Guide를 반드시 반영하라
- Tone Guide의 어조(존칭/반말, 정중/캐주얼 등)를 정확히 따라야 한다
- Stage(할인/신제품런칭/재구매 등)에 맞는 메시지 방향성을 유지하라
- Vibe(2030/4060)에 맞는 표현 수준을 사용하라

### 3) Event 정합성 검증
- 컨텍스트의 Event 항목을 반드시 확인하라
- Event가 "없음"이면: 본문
[GPT] user_preview=
[컨텍스트]
- Persona: 프리미엄 럭셔리 스마트 자산가
- Stage: 베스트셀러
- Brand/Product: 아이오페 / 슈퍼바이탈 레티놀 세럼
- Event: 없음
- Vibe: 4060
- Tone Guide: 존칭 중심의 정중한 문장으로 안정감과 신뢰를 전달하며, 차분한 표현으로 품질과 편의성을 강조합니다.
- Context Example: 유튜브에서 고가 화장품의 성분 분석 리뷰를 보며 구매 의사를 굳힐 때

### [슈퍼바이탈 레티놀 세럼] 

#### [본문] 
속부터 깊은 탄력과 고효율 레티놀로 피부 건강을 한 단계 업그레이드하세요. 민감피부도 안심하고 사용 가능한 순한 제형이 특징이며, 과학적으로 입증된 성분들이 피부 장벽 강화와 주름 개선을 동시에 지원합니다. 수분 밸런스 회복과 함께 자연스러운 광채 연출까지 한 번에 해결해 줍니다. 

프리미엄 라인으로 개발된 이 제품은 지속적인 사용 시 피부 대사율 향상과 노화 방지 효과를 확인할 수 있습니다. 기존 제
[GPT] output_text_len=167
[GPT] output_text_preview=
[제목] 피부 탄력을 위한 선택, 레티놀 세럼
[본문] 깊은 탄력과 피부 건강을 위한 최적의 선택, 아이오페의 슈퍼바이탈 레